# Phase 2: Train the GNN on Google Colab

This notebook trains the Phase 2 dynamic-topology GNN on Google Colab's free tier.

The model is small (11-node graph x 14 features x GCNConv(32->16->1) + centrality head) so it trains in seconds on CPU and stays well within Colab's free-tier RAM/timeout caps.

**Inputs** (from your Google Drive, after you put them there):

- `MyDrive/aizen/db/aizen.db` — the SQLite system of record (with `v_features_underlying_v2` and `v_labels` views populated).

**Outputs** (written back to Drive):

- `MyDrive/aizen/models/gnn-YYYYMMDD-NNNN.pt`
- `MyDrive/aizen/models/gnn-YYYYMMDD-NNNN.meta.json`
- `MyDrive/aizen/models/report_phase2_gnn.json`

After the run, copy the two artifacts into your local repo's `models/` directory and the orchestrator will pick the new model up automatically via `GNNService.load_latest(conn)`.

## 1. Install dependencies (CPU-only torch is the fastest on free Colab)

In [ ]:
%%capture
import sys
print(f"Python: {sys.version}")

# CPU-only torch is ~200MB smaller than the CUDA build and avoids the
# 1-2 minute CUDA download on a fresh Colab VM. PyG works fine on CPU
# for an 11-node graph.
%pip install --quiet torch==2.4.0 --index-url https://download.pytorch.org/whl/cpu
%pip install --quiet -r https://raw.githubusercontent.com/<YOUR-ORG>/<YOUR-REPO>/main/requirements.txt

import torch, torch_geometric, jsonschema
print(f"torch: {torch.__version__}  |  PyG: {torch_geometric.__version__}  |  cuda: {torch.cuda.is_available()}")

## 2. Mount Drive and stage the repo + DB

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# --- paths you may want to edit -----------------------------------------
DRIVE_REPO_DIR  = '/content/drive/MyDrive/aizen'         # where the repo lives on Drive
DRIVE_DB_PATH   = f'{DRIVE_REPO_DIR}/db/aizen.db'        # the SQLite system of record
DRIVE_MODELS_DIR = f'{DRIVE_REPO_DIR}/models'            # where artifacts are written
# ------------------------------------------------------------------------

import os, pathlib, shutil

REPO_DIR = '/content/aizen-trading'

# Option A: clone from GitHub (recommended for the first run).
# Edit the URL below to match your fork.
REPO_URL = 'https://github.com/<YOUR-ORG>/<YOUR-REPO>.git'  # <-- change me
if not pathlib.Path(REPO_DIR).exists():
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print(f'repo already present at {REPO_DIR}; pulling latest')
    %cd {REPO_DIR}
    !git pull --ff-only

%cd {REPO_DIR}

# Stage a local copy of the DB so we don't hammer Drive over the
# full training run. The training loop re-reads the DB for every
# snapshot, so a local copy is much faster than reading across the
# FUSE mount.
assert pathlib.Path(DRIVE_DB_PATH).exists(), (
    f'database not found at {DRIVE_DB_PATH}. '
    'Upload aizen.db to that location and re-run this cell.'
)
shutil.copy2(DRIVE_DB_PATH, '/content/aizen.db')
DB_PATH = '/content/aizen.db'
print(f'using DB at {DB_PATH}  ({pathlib.Path(DB_PATH).stat().st_size:,} bytes)')

## 3. Verify the dataset has enough snapshots to train

In [ ]:
import sqlite3
from src.gnn.constants import UNIVERSE

conn = sqlite3.connect(DB_PATH)
ph = ','.join('?' * len(UNIVERSE))
n = conn.execute(
    f"SELECT COUNT(DISTINCT timestamp) FROM v_labels "
    f"WHERE horizon_bars=16 AND symbol IN ({ph}) "
    f"AND target_class IS NOT NULL AND target_class != 0",
    list(UNIVERSE),
).fetchone()[0]
print(f'available training snapshots (horizon=16, non-flat): {n}')
assert n >= 3, 'need at least 3 snapshots; build the feature dataset first'
conn.close()

## 4. Train the GNN

In [ ]:
# Train with 100 epochs, write artifacts to a local dir first
# (faster than writing across the FUSE mount), then copy to Drive.
import os, json, pathlib, time, subprocess

LOCAL_OUT_DIR = '/content/gnn_models'
pathlib.Path(LOCAL_OUT_DIR).mkdir(parents=True, exist_ok=True)

cmd = [
    'python', '-m', 'src.gnn.train',
    '--db-path', DB_PATH,
    '--n-snapshots', '50',
    '--epochs', '100',
    '--architecture', 'gcn-32-16-1',
    '--out-dir', LOCAL_OUT_DIR,
    '--out-prefix', 'gnn',
]
print(' '.join(cmd))
t0 = time.time()
result = subprocess.run(cmd, capture_output=True, text=True, check=False)
print('--- stdout ---')
print(result.stdout)
if result.returncode != 0:
    print('--- stderr ---')
    print(result.stderr)
    raise SystemExit(result.returncode)
print(f'\ntotal wall time: {time.time() - t0:.1f}s')

result = json.loads(result.stdout)
print('\nFinal test metrics:')
for k, v in result['test_metrics'].items():
    print(f'  {k:>9}: {v:.4f}')

## 5. Copy artifacts to Drive

In [ ]:
import shutil, os, pathlib

pathlib.Path(DRIVE_MODELS_DIR).mkdir(parents=True, exist_ok=True)

shutil.copy2(result['artifact_path'], DRIVE_MODELS_DIR)
shutil.copy2(result['meta_path'],    DRIVE_MODELS_DIR)

print('Copied to Drive:')
for f in (result['artifact_path'], result['meta_path']):
    src = pathlib.Path(f)
    dst = pathlib.Path(DRIVE_MODELS_DIR) / src.name
    print(f'  {dst}  ({dst.stat().st_size:,} bytes)')

print('\nmodel_version:', result['model_version'])

## 6. (Optional) Generate the side-by-side report

In [ ]:
# Re-evaluates the saved artifact on the same chronological split and
# writes models/report_phase2_gnn.json (Phase 1 XGB baseline vs. Phase 2 GNN).
# Then copy that report to Drive too.
import subprocess, json, pathlib, shutil

code = (
    "from src.gnn.evaluate import evaluate_model; "
    f"r = evaluate_model(__import__('sqlite3').connect('{DB_PATH}'), "
    f"'{result['artifact_path']}', "
    "n_snapshots=50, "
    f"report_path='{LOCAL_OUT_DIR}/report_phase2_gnn.json'); "
    "import json; print(json.dumps(r, indent=2))"
)
out = subprocess.run(['python', '-c', code], capture_output=True, text=True)
print(out.stdout)
if out.returncode != 0:
    print(out.stderr); raise SystemExit(out.returncode)

shutil.copy2(
    f'{LOCAL_OUT_DIR}/report_phase2_gnn.json',
    f'{DRIVE_MODELS_DIR}/report_phase2_gnn.json',
)
print(f'\nReport saved to {DRIVE_MODELS_DIR}/report_phase2_gnn.json')

## 7. Back on your local machine

After the run, the artifacts are sitting in `MyDrive/aizen/models/`. Pull them into your local repo:

```bash
# from the repo root, with Drive synced or rclone-mounted
cp "/mnt/c/Users/<you>/Google Drive/aizen/models/gnn-$(date +%Y%m%d)-NNNN.pt"      models/
cp "/mnt/c/Users/<you>/Google Drive/aizen/models/gnn-$(date +%Y%m%d)-NNNN.meta.json" models/
cp "/mnt/c/Users/<you>/Google Drive/aizen/models/report_phase2_gnn.json"             models/
```

Then verify the orchestrator picks up the new model:

```bash
python -c "
import sqlite3, json
from src.agents.inference import InferenceService
conn = sqlite3.connect('db/aizen.db')
svc = InferenceService(conn, universe=['SPY','QQQ','AAPL','MSFT','NVDA','AMZN','META','GOOGL','TSLA','AMD'])
out = svc.gnn_output()
print('model_version:', out['model_version'])
print('contract-valid:', bool(out['node_features']) and bool(out['edges']))
" 
```

The `model_version` should match the new artifact (`gnn-YYYYMMDD-NNNN`) and `gnn_output_json` in the decision journal will reference it from then on.